In [3]:
import pandas as pd

# --- Step 1: List your monthly Excel files ---
# Make sure all files are in the same folder as this script or notebook
files = [
    "QUICKMART DATA JULY.xlsx",
    "QUICKMART DATA AUGUST.xlsx",
    "QUICKMART DATA SEPTEMBER.xlsx",
    "QUICKMART DATA OCTOBER.xlsx",
    "QUICKMART DATA NOVEMBER.xlsx"
]

# --- Step 2: Read and store each file into a list ---
dataframes = []
for file in files:
    df = pd.read_excel(file)
    
    # Optional: Add a 'Month' column if not already present
    # Extract month name from the file name automatically
    month_name = file.split("DATA")[1].replace(".xlsx", "").strip()
    df["Month"] = month_name
    
    dataframes.append(df)

# --- Step 3: Append (concatenate) all monthly data together ---
combined_quickmart = pd.concat(dataframes, ignore_index=True)

# --- Step 4: Save to a single Excel file for further analysis ---
combined_quickmart.to_excel("QUICKMART_COMBINED.xlsx", index=False)

print("✅ Combined Quickmart data saved as 'QUICKMART_COMBINED.xlsx'")
print(f"Total rows combined: {len(combined_quickmart)}")
print("Columns:", list(combined_quickmart.columns))


✅ Combined Quickmart data saved as 'QUICKMART_COMBINED.xlsx'
Total rows combined: 489604
Columns: ['STORE_NAME', 'ITEM_NAME', 'QUANTITY', 'TOTAL SALES', 'SUPPLIER', 'CATEGORY', 'DEPARTMENT', 'SUB DEPARTMENT', 'MICRO DEPARTMENT', 'Month']


In [4]:
import pandas as pd
import re

# --- Load your dataset ---
df = pd.read_excel("QUICKMART_COMBINED.xlsx")

def fix_micro_department(row):
    # capture original raw micro (uppercase/stripped) for rules that depend on original value
    orig_micro = str(row['MICRO DEPARTMENT']).strip().upper()
    micro = orig_micro  # we'll update micro with grouped values later
    item = str(row['ITEM_NAME']).strip().upper()

 # Antibacterial logic: if original micro was VALUE HANDWASH and item matches pattern
    if re.search(r"A/?B(ACT|AC)", item) and orig_micro == "VALUE HANDWASH":
        return "ANTIBACTERIAL HANDWASH"

    # Toilet balls fix (original micro)
    if ("BALLS" in item or "BALL" in item) and orig_micro == "TOILET BLOCK":
        return "TOILET BALL"

    # Glass logic with DISINFECTANT original micro
    if "GLASS" in item and orig_micro == "DISINFECTANT":
        return "GLASS CLEANER"

    # Glue correction (original micro)
    if "GLUE" in item and orig_micro == "TAPES AND ADHESIVES":
        return "GLUE"

    # Waterguard (original micro)
    if "AQUAGUARD" in item and orig_micro == "HAND SANITIZER":
        return "WATERGUARD"

    # Window cleaner fix if original was MULTIPURPOSE CLEANER
    if "HC-TOPEX WINDOW CLEANER 300ML LAV" in item:
      return "WINDOW CLEANER"

    if "WINDOW" in item and orig_micro == "MULTIPURPOSE CLEANER":
        return "GLASS CLEANER"

    # BODY WASH special-case that depended on original SOCIAL HANDWASH
    if "BODY WASH" in item and orig_micro == "SOCIAL HANDWASH":
        return "BODY WASH"

    # --------------------
    # NEW / GROUPING RULES (apply after original-specific rules)
    # --------------------

    # 1. Concentrate + Dilute → FABRIC SOFTENERS
    if orig_micro in ["CONCENTRATE", "DILUTE"]:
        return "FABRIC SOFTENERS"

    # 2. Scented + Value Pack → REGULAR
    if orig_micro in ["SCENTED", "VALUE PACK"]:
        return "REGULAR"

    # 3. Shower Gels (both family & beauty) → BODY WASH
    if orig_micro in ["SHOWER GELS FAMILY", "SHOWER GELS BEAUTY"]:
        return "BODY WASH"

    # 4. Value Handwash + Social Handwash → HANDWASH
    # (we already handled the special antibacterial & BODY WASH cases above using orig_micro)
    if orig_micro in ["VALUE HANDWASH", "SOCIAL HANDWASH"]:
        return "HANDWASH"

    # 5. Dishwashing Liquid → MULTIPURPOSE DISHWASHING based on keywords
    if orig_micro == "DISHWASHING LIQUID":
        if any(x in item for x in ["LIQUID DETERGENT", "LIQUIDDETERGENT", "MULTIPURPOSE",
                                   "M/PURP", "M/P", "M/PURPOSE", "MULTI-PURPOSE", "MDWL"]):
            return "MULTIPURPOSE DISHWASHING"

    # 6. Glass Cleaner → WINDOW CLEANER if 300ml appears
    if orig_micro == "GLASS CLEANER":
        if "TOPEX" in item:
            return "WINDOW CLEANER"
        # otherwise keep as GLASS CLEANER
        return "GLASS CLEANER"

    # 7. Hand Sanitizer → HANDWASH if h/wash exists (also keep sanitizer if not)
    if orig_micro == "HAND SANITIZER":
        if "H/WASH" in item or "H/WASH" in item.replace(" ", ""):
            return "HANDWASH"
        return "HAND SANITIZER"  # keep as sanitizer if not matched
 
    # 8. Kitchen Cleaner → HOME CLEANING if keywords match
    if orig_micro == "KITCHEN CLEANER":
        if any(x in item for x in ["WASHING", "W/MACH", "WMACH", "FRIDGE", "OVEN", "REMOVER", "MOLD", "STAIN"]):
            return "HOME CLEANING"
        return "KITCHEN CLEANER"

    # 9. Multipurpose Cleaning → various mappings
    if orig_micro == "MULTIPURPOSE CLEANER":
        if any(x in item for x in ["W/LIQUID", "W LIQUID", "WASHING LIQUID", "WASHINGLIQUID"]):
            return "MULTIPURPOSE DISHWASHING"
        if "HOME DRY" in item:
            return "HOME DRY CLEANING"
        if "GLASS" in item:
            return "GLASS CLEANER"
        return "MULTIPURPOSE CLEANER"

    # --------------------
    # DEFAULT: if none of the above matched, return the original (cleaned) micro
    # --------------------
    return orig_micro

# --- Apply the cleaning ---
df['MICRO DEPARTMENT'] = df.apply(fix_micro_department, axis=1)

# --- Save output ---
df.to_excel("QUICKMART_COMBINED_CLEANED.xlsx", index=False)

print("✅ Cleaning complete! File saved as QUICKMART_COMBINED_CLEANED.xlsx")


✅ Cleaning complete! File saved as QUICKMART_COMBINED_CLEANED.xlsx


In [5]:
import pandas as pd

# --- STEP 1: Load your data file (with branch list) ---
data = pd.read_excel("QUICKMART_COMBINED_CLEANED.xlsx")

# --- STEP 2: Clean the branch names for reliable matching ---
data["STORE_NAME_CLEAN"] = data["STORE_NAME"].str.upper().str.strip()

# --- STEP 3: Define your mapping list (dictionary) ---
branch_handler_map = {
    "BANDARI": "VERONICA ANYONA WOCHETE",
    "BURUBURU": "VERONICA MWENDE MAWEU",
    "CHAKA RD": "CECILIA WANJIRU",
    "CHANIA": "MARGARET NJERI",
    "CROSSROADS-KAREN": "JOEL SIMIYU",
    "DONHOLM": "CYNTHIA ANYANGO",
    "EASTERN BYPASS 1": "ISABELA WAMBUI",
    "EASTERN BYPASS 2": "ISABELA WAMBUI",
    "ELDOCENTER": "ELISHA JUMA WAFULA",
    "EMBAKASI": "MERCY MUTANU",
    "FEDHA": "CYNTHIA ANYANGO",
    "HIGHWAY-MLOLONGO": "ANN LOLE",
    "JIPANGE": "CLEFTON WAMBUA MUENDO",
    "JOSKA": "EMILY MORAA",
    "KAHAWA SUKARI": "LINET ATIENO BELLO",
    "KAHAWA WEST": "ISABELA WAMBUI",
    "KAKAMEGA": "JOSEPH NDETEI MAKAU",
    "KERICHO" : "BONIFACE ONKWAMI",
    "KIAMBU RD": "JULIET NDUTA MBURU",
    "KIKUYU RD": "SHAMITA MAKENA",
    "KILELESHWA": "SHAMITA MAKENA",
    "KILIMANI": "CECILIA WANJIRU",
    "KISERIAN": "JOEL SIMIYU",
    "KISII": "OLIPHA NYANGAMI",
    "KITALE": "FREDRICK OMONDI YADA",
    "KITENGELA": "PERIS WAVINYA KITAKA",
    "KITUI": "PERIS WAVINYA KITAKA",
    "KONDELE": "WALUBENGO VICTOR", 
    "LAVINGTON": "SHAMITA MOHAMED",
    "MACHAKOS EXPRESS": "PERIS WAVINYA KITAKA",
    "MACHAKOS PIONEER": "PERIS WAVINYA KITAKA",
    "MAYFAIR": "DORCAS NJERI KIMANI",
    "MFANGANO": "STEPHEN OTIENO",
    "MILELE-NGONG": "JOEL SIMIYU",
    "MOMBASA ROAD": "JAMES KARUGU WAITHERA",
    "MTWAPA EXPRESS": "ELIAS NDWIGA",
    "MTWAPA MALL": "ELIAS NDWIGA",
    "NANYUKI":"VERONICA WACUKA",
    "NYALENDA": "WALUBENGO VICTOR",
    "NYALI": "ELIAS NDWIGA",
    "OGINGA ODINGA": "DORCAS NJERI KIMANI",
    "OTC": "STEPHEN OTIENO",
    "OUTERING": "CYNTHIA ANYANGO",
    "PIONEER": "STEPHEN OTIENO",
    "PIPELINE": "MERCY MUTANU",
    "RONGAI EXPRESS": "IRENE NTHENYA MUTHOKA",
    "RONGAI MAIN": "IRENE NTHENYA MUTHOKA",
    "ROYSAMBU": "CLEFTON WAMBUA MUENDO",
    "RUAI": "EMILY MORAA",
    "RUAKA": "ELIZABETH MUMBUA",
    "RUIRU": "PAUL GUCHU",
    "SHABAAB": "BEATRICE ANYANGO",
    "T-MALL": "DENNIS KIPRUTO",
    "TOM MBOYA": "ELIZABETH MUMBUA",
    "UTAWALA EXPRESS": "MERCY MUTANU",
    "UTAWALA MAIN": "MERCY MUTANU",
    "WAIYAKI WAY": "ELIZABETH MUMBUA",
    "WESTLANDS": "ELIZABETH MUMBUA"
}

# --- STEP 4: Map handlers to the cleaned branch names ---
data["HANDLER"] = data["STORE_NAME_CLEAN"].map(branch_handler_map).fillna("NO MATCH FOUND")

# --- STEP 5: Save result ---
data.to_excel("QUICKMART_WITH_HANDLERS.xlsx", index=False)

print("✅ Handlers successfully mapped and saved to 'QUICKMART_WITH_HANDLERS.xlsx'")


✅ Handlers successfully mapped and saved to 'QUICKMART_WITH_HANDLERS.xlsx'


In [6]:
import pandas as pd

# --- STEP 1: Load your branches file (from previous step) ---
data = pd.read_excel("QUICKMART_WITH_HANDLERS.xlsx")

# --- STEP 2: Define branch-to-region mapping ---
region_map = {
    # Nairobi Region
    "BURUBURU": "NAIROBI",
    "CHAKA RD": "NAIROBI",
    "CHANIA": "NAIROBI",
    "CIRCLE MALL": "NAIROBI",
    "CROSSROADS-KAREN": "NAIROBI",
    "DONHOLM": "NAIROBI",
    "EASTERN BYPASS 1": "NAIROBI",
    "EASTERN BYPASS 2": "NAIROBI",
    "EMBAKASI": "NAIROBI",
    "FEDHA": "NAIROBI",
    "HIGHWAY-MLOLONGO": "NAIROBI",
    "JIPANGE": "NAIROBI",
    "JOSKA": "NAIROBI",
    "KAHAWA SUKARI": "NAIROBI",
    "KAHAWA WEST": "NAIROBI",
    "KIAMBU RD": "NAIROBI",
    "KIKUYU RD": "NAIROBI",
    "KILELESHWA": "NAIROBI",
    "KILIMANI": "NAIROBI",
    "LAVINGTON": "NAIROBI",
    "MAYFAIR": "NAIROBI",
    "MFANGANO": "NAIROBI",
    "MILELE-NGONG": "NAIROBI",
    "MOMBASA ROAD": "NAIROBI",
    "OTC": "NAIROBI",
    "OUTERING": "NAIROBI",
    "OUTERING 2": "NAIROBI",
    "PIONEER": "NAIROBI",
    "PIPELINE": "NAIROBI",
    "RONGAI EXPRESS": "NAIROBI",
    "RONGAI MAIN": "NAIROBI",
    "ROYSAMBU": "NAIROBI",
    "RUAI": "NAIROBI",
    "RUAKA": "NAIROBI",
    "RUIRU": "NAIROBI",
    "THOME": "NAIROBI",
    "T-MALL": "NAIROBI",
    "TOM MBOYA": "NAIROBI",
    "UTAWALA EXPRESS": "NAIROBI",
    "UTAWALA MAIN": "NAIROBI",
    "WAIYAKI WAY": "NAIROBI",
    "WESTLANDS": "NAIROBI",

    # Coastal Region
    "BANDARI": "COAST",
    "MTWAPA EXPRESS": "COAST",
    "MTWAPA MALL": "COAST",
    "NYALI": "COAST",

    # Eastern Region
    "KITENGELA": "EASTERN",
    "KITUI": "EASTERN",
    "MACHAKOS EXPRESS": "EASTERN",
    "MACHAKOS PIONEER": "EASTERN",

    # Western Region
    "KAKAMEGA": "WESTERN",
    "KISII": "WESTERN",
    "KITALE": "WESTERN",
    "KONDELE": "WESTERN",
    "NYALENDA": "WESTERN",
    "OGINGA ODINGA": "WESTERN",
    "ELDOCENTER": "WESTERN",

    # Central Region
    "CHAKA RD": "CENTRAL",
    "KIAMBU RD": "CENTRAL",
    "KIKUYU RD": "CENTRAL",
    "NANYUKI": "CENTRAL",

    # Rift Valley Region
    "KERICHO": "RIFT VALLEY",
    "KISERIAN": "RIFT VALLEY",
    "NAROK MORAN": "RIFT VALLEY",
    "SHABAAB": "RIFT VALLEY"
    
}

# --- STEP 3: Map regions to branches ---
data["Region"] = data["STORE_NAME_CLEAN"].map(region_map).fillna("No Region Found")

# --- STEP 4: Save updated file ---
data.to_excel("QUICKMART_final_data.xlsx", index=False)

print("✅ Regions successfully mapped and saved to 'branches_with_handlers_and_regions.xlsx'")
print(data.head(10))

✅ Regions successfully mapped and saved to 'branches_with_handlers_and_regions.xlsx'
  STORE_NAME                                ITEM_NAME  QUANTITY   TOTAL SALES  \
0    SHABAAB       HH-QNP STANDARD SATURDAY NEWSPAPER       7.0    362.068687   
1    SHABAAB      HH-QNP NSS BUSINESS DAILY NEWSPAPER       2.0    172.413803   
2    SHABAAB               HC-UNI VIM LEMON FRESH 1KG       8.0   1310.344849   
3    SHABAAB                HH-QNP THE STAR NEWSPAPER       3.0    155.172295   
4    SHABAAB                     HC-TOPEX LEMON 500ML       1.0    198.275803   
5    SHABAAB        HC-CENTURY HOME DRY CLEANER 500ML      13.0   3362.067871   
6    SHABAAB  HC-SCJ KIWI SHOE POLISH BLACK 40ML/50ML     141.0  16163.787300   
7    SHABAAB       HC-SCJ KIWI POLISH BLACK 100ML/80g      94.0  21374.131989   
8    SHABAAB            HC-CAN PRIDE PASTE LEMON 400G      12.0   2586.206345   
9    SHABAAB     HC-SBNSAFISHA DWPASTE LEMON500G/400G       4.0    727.585999   

                       